# Phase 03.00 — LoRA rank ablation r={8,16,32,64}

This notebook trains and evaluates one independent F1 adapter per LoRA rank. Only rank and the derived alpha change; dataset membership, seed, target modules, frame policy, optimizer, epochs, masking, and attention implementation remain fixed.


In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)


Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


## 1. Controlled configuration

The default is a two-step rank-8 smoke run. Set `DEBUG_MODE=False` only after Phase 01 and Phase 02 full runs pass.


In [2]:
import gc, time
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model

DEBUG_MODE = True
RANKS = [8] if DEBUG_MODE else [8, 16, 32, 64]
TRAIN_LIMIT = 20 if DEBUG_MODE else None
EVAL_LIMIT = 20 if DEBUG_MODE else None
MAX_STEPS = 2 if DEBUG_MODE else None
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

OUTPUT_ROOT = PROJECT_ROOT / "outputs/phase03/lora_rank_ablation"
train_df = pd.read_csv(PATHS.phase1_split / "train.csv")
val_df = pd.read_csv(PATHS.phase1_split / "validation.csv")
frozen_ids = json.loads((PATHS.phase1_split / "validation_sample_ids.json").read_text(encoding="utf-8"))
assert sorted(val_df.sample_id.astype(str)) == frozen_ids
run_scope = "debug20" if DEBUG_MODE else "full"


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Train/evaluate one rank

The base model is reloaded for every rank to prevent adapter or optimizer state leakage. Each run saves its own adapter, training history, predictions, metrics, and configuration.


In [3]:
def train_and_evaluate_rank(rank):
    seed_everything(SEED)
    started = time.perf_counter()
    model, tokenizer = load_model_and_tokenizer(training=True)
    model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
    for parameter in model.parameters():
        parameter.requires_grad = False

    config = LoraConfig(
        r=rank, lora_alpha=2 * rank, lora_dropout=0.05, bias="none",
        target_modules=TARGET_MODULES,
    )
    model = get_peft_model(model, config)
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    assert trainable > 0
    assert all(parameter.requires_grad == ("lora_" in name) for name, parameter in model.named_parameters())

    dataset = RoadBuddySFTDataset(train_df, model, tokenizer, limit=TRAIN_LIMIT)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        collate_fn=make_collator(tokenizer), generator=torch.Generator().manual_seed(SEED),
    )
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history, optimizer_step = [], 0
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(EPOCHS):
        for micro_step, batch in enumerate(loader, 1):
            sample_ids = batch.pop("sample_ids")
            batch = {key: value.cuda(non_blocking=True) for key, value in batch.items()}
            assert batch["image_flags"].numel() == batch["pixel_values"].shape[0]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = model(**batch).loss / GRADIENT_ACCUMULATION
            loss.backward()
            if micro_step % GRADIENT_ACCUMULATION == 0 or micro_step == len(loader):
                torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), MAX_GRAD_NORM)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_step += 1
                history.append({"rank": rank, "epoch": epoch + 1, "step": optimizer_step, "loss": float(loss.item() * GRADIENT_ACCUMULATION), "sample_ids": sample_ids})
                print(history[-1])
                if MAX_STEPS is not None and optimizer_step >= MAX_STEPS:
                    break
        if MAX_STEPS is not None and optimizer_step >= MAX_STEPS:
            break

    out_dir = OUTPUT_ROOT / f"r{rank}" / run_scope
    adapter_dir = out_dir / "checkpoints/adapter_final"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    pd.DataFrame(history).to_csv(out_dir / "train_history.csv", index=False)

    model.eval()
    predictions, metrics = evaluate_rows(model, tokenizer, val_df, limit=EVAL_LIMIT)
    eval_dir = out_dir / "evaluation"
    eval_dir.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(eval_dir / "validation_predictions.csv", index=False)
    elapsed = time.perf_counter() - started
    result = {
        "rank": rank, "lora_alpha": 2 * rank, "trainable_parameters": trainable,
        "optimizer_steps": optimizer_step, "peak_vram_gib": torch.cuda.max_memory_allocated() / 1024**3,
        "elapsed_seconds": elapsed, **metrics,
    }
    save_json(eval_dir / "validation_metrics.json", result)
    save_json(out_dir / "config.json", {
        "model_id": MODEL_ID, "revision": MODEL_REVISION, "seed": SEED,
        "rank": rank, "lora_alpha": 2 * rank, "lora_dropout": 0.05,
        "target_modules": TARGET_MODULES, "frames": 1, "template": "Hermes-2",
        "label_masking": "token_prefix", "image_flags": "one_per_visual_tile",
        "flash_attention": False, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRADIENT_ACCUMULATION, "learning_rate": LEARNING_RATE,
        "debug_mode": DEBUG_MODE, "max_steps": MAX_STEPS,
    })
    del model, optimizer, dataset, loader
    gc.collect()
    torch.cuda.empty_cache()
    return result


## 3. Run the rank grid


In [4]:
rank_results = []
for rank in RANKS:
    print(f"\n===== LoRA rank {rank} =====")
    rank_results.append(train_and_evaluate_rank(rank))

summary = pd.DataFrame(rank_results).sort_values("rank")
summary.to_csv(OUTPUT_ROOT / f"phase03_rank_summary_{run_scope}.csv", index=False)
display(summary)



===== LoRA rank 8 =====


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


,rank,lora_alpha,trainable_parameters,optimizer_steps,peak_vram_gib,elapsed_seconds,rows,accuracy,macro_f1,parse_rate
0,8,16,1081344,2,14.228646,44.680237,20,0.65,0.598485,1.0


## 4. Full-run integrity and status


In [5]:
expected_ranks = set(RANKS)
assert set(summary["rank"]) == expected_ranks
expected_rows = min(len(val_df), EVAL_LIMIT) if EVAL_LIMIT else len(val_df)
assert summary["rows"].eq(expected_rows).all()
best = summary.sort_values(["accuracy", "macro_f1"], ascending=False).iloc[0]
phase_status = "debug_complete" if DEBUG_MODE else "complete"
save_json(OUTPUT_ROOT / "PHASE03_STATUS.json", {
    "phase": "03", "status": phase_status, "run_name": run_scope,
    "ranks": sorted(expected_ranks), "best_rank": int(best["rank"]),
    "best_accuracy": float(best.accuracy), "validation_rows": expected_rows,
    "model_revision": MODEL_REVISION, "seed": SEED,
})
print(f"Phase 03 {run_scope}: {phase_status.upper()}")

Phase 03 debug20: DEBUG_COMPLETE
